In [1]:
# ==========================================================
# BLOCK 1: BINARY SETUP & DATA LOADING
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# 1. Global Reproducibility (Q1 Journal Requirement)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. GPU Setup (Safe Memory Growth)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs Detected: {len(gpus)}")
    except RuntimeError as e:
        print(e)

# 3. Directories & Constants
DATASET_ROOT = '/home/T2430514/Downloads/MargeDataset/Binary'
ANOMALY_DIR = os.path.join(DATASET_ROOT, 'Anomaly')
NORMAL_DIR = os.path.join(DATASET_ROOT, 'Normal')
PROCESSED_DATA_DIR = '/home/T2430514/Downloads/MargeDataset/Processed' 
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

FRAME_SIZE = (224, 224)
NUM_FRAMES = 16
BATCH_SIZE = 4 

# 4. Data Loading Logic
def create_dataframe():
    data = []
    # Anomaly (Label 1)
    if os.path.exists(ANOMALY_DIR):
        for video_file in os.listdir(ANOMALY_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(ANOMALY_DIR, video_file), 'bin_label': 1})
    
    # Normal (Label 0)
    if os.path.exists(NORMAL_DIR):
        for video_file in os.listdir(NORMAL_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(NORMAL_DIR, video_file), 'bin_label': 0})
                
    return pd.DataFrame(data)

all_df = create_dataframe()
print(f"Total Binary Videos: {len(all_df)}")

# 5. Stratified Split (Crucial for Imbalanced/Small Data)
train_df, temp_df = train_test_split(all_df, test_size=0.2, stratify=all_df['bin_label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['bin_label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 6. Compute Class Weights
y_train = train_df['bin_label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weights_dict}")

2026-04-26 20:41:56.740535: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-26 20:41:56.746883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777214516.755204 3824320 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777214516.757863 3824320 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777214516.764289 3824320 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPUs Detected: 1
Total Binary Videos: 4534
Train: 3627 | Val: 453 | Test: 454
Class Weights: {0: np.float64(1.0002757859900717), 1: np.float64(0.9997243660418964)}


In [2]:
# ==========================================================
# BLOCK 2: FRAME EXTRACTION
# ==========================================================
import sys

def extract_and_save_frames(dataframe, output_dir, num_frames=NUM_FRAMES, frame_size=FRAME_SIZE):
    print(f"Processing {len(dataframe)} videos...")
    count = 0
    
    for idx, row in dataframe.iterrows():
        base_name = os.path.basename(row.path)
        # Use .npy for faster loading during training
        save_name = os.path.splitext(base_name)[0] + '.npy'
        output_path = os.path.join(output_dir, save_name)
        
        # Skip if already processed
        if os.path.exists(output_path): 
            continue

        cap = cv2.VideoCapture(row.path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            cap.release()
            continue
            
        # Uniform Temporal Sampling (SOTA standard)
        frame_indices = np.linspace(0, max(total_frames - 1, 0), num=num_frames, dtype=int)
        frames = []
        
        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, frame_size)
                frames.append(frame)
            else:
                # Padding if read fails
                frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
        cap.release()
        
        # Ensure exact frame count
        while len(frames) < num_frames:
            frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
            
        # Save as uint8 to save disk space (converted to float32 in generator)
        np.save(output_path, np.array(frames, dtype=np.uint8))
        
        count += 1
        if count % 100 == 0: 
            sys.stdout.write(f"\rExtracted {count} videos.")
    print("\nExtraction Complete.")

extract_and_save_frames(all_df, PROCESSED_DATA_DIR)

Processing 4534 videos...

Extraction Complete.


In [3]:
# ==========================================================
# BLOCK 3: SOTA DATA GENERATOR (EXPERIMENT B: MIXUP)
# ==========================================================
import tensorflow as tf
import numpy as np
import os

class MixUpVideoDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, processed_data_dir, batch_size=BATCH_SIZE, 
                 num_frames=NUM_FRAMES, frame_size=FRAME_SIZE, 
                 augment=False, shuffle=True, mixup_alpha=0.3):
        self.dataframe = dataframe
        self.processed_data_dir = processed_data_dir
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.augment = augment
        self.shuffle = shuffle
        self.mixup_alpha = mixup_alpha 
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        return self.__data_generation(batch_indices)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_video(self, video_path):
        base_name = os.path.basename(video_path)
        name, _ = os.path.splitext(base_name)
        npy_path = os.path.join(self.processed_data_dir, name + '.npy')
        
        if os.path.exists(npy_path):
            try: 
                return np.load(npy_path).astype(np.float32) / 255.0
            except: 
                pass
        return np.zeros((self.num_frames, *self.frame_size, 3), dtype=np.float32)

    def apply_consistent_augmentation(self, video):
        """
        Applies Memory-Safe Augmentations using pure NumPy.
        Prevents TensorFlow EagerTensor RAM leaks.
        """
        # 1. Random Horizontal Flip (axis=2 is the width dimension)
        if np.random.rand() > 0.5:
            video = np.flip(video, axis=2)
            
        # 2. Random Brightness (-0.15 to 0.15)
        brightness_delta = np.random.uniform(-0.15, 0.15)
        video = video + brightness_delta
        
        # 3. Random Contrast (0.85 to 1.15)
        contrast_factor = np.random.uniform(0.85, 1.15)
        mean = np.mean(video, axis=(1, 2, 3), keepdims=True)
        video = (video - mean) * contrast_factor + mean
        
        # Clip strictly to [0.0, 1.0] and ensure float32
        return np.clip(video, 0.0, 1.0).astype(np.float32)

    def __data_generation(self, batch_indices):
        X = np.empty((self.batch_size, self.num_frames, *self.frame_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size), dtype=np.float32) 

        for i, idx in enumerate(batch_indices):
            row = self.dataframe.iloc[idx]
            video = self.load_video(row.path)
            label = float(row.bin_label)

            # --- MIXUP LOGIC ---
            if self.augment and self.mixup_alpha > 0 and np.random.rand() < 0.5:
                rand_idx = np.random.choice(self.indices)
                row2 = self.dataframe.iloc[rand_idx]
                video2 = self.load_video(row2.path)
                label2 = float(row2.bin_label)

                # Sample lambda from Beta distribution
                lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
                
                # Interpolate inputs and targets
                video = lam * video + (1 - lam) * video2
                label = lam * label + (1 - lam) * label2

            # --- SPATIAL AUGMENTATION ---
            if self.augment:
                video = self.apply_consistent_augmentation(video)

            X[i,] = video
            y[i] = label

        return X, y
    
    def get_labels(self):
        original_indices = self.indices.copy()
        if self.shuffle:
            sorted_indices = np.arange(len(self.dataframe))
        else:
            sorted_indices = self.indices
            
        limit = self.__len__() * self.batch_size
        return self.dataframe.iloc[sorted_indices[:limit]]['bin_label'].values

# Alias for model blocks to use seamlessly
SOTAVideoDataGenerator = MixUpVideoDataGenerator

print("Initializing Generators (Experiment B: MixUp Augmentation - NUMPY FIX)...")
train_generator = SOTAVideoDataGenerator(
    train_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=True, mixup_alpha=0.3, shuffle=True
)
val_generator = SOTAVideoDataGenerator(
    val_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False
)
print("MixUp Generators Ready.")

Initializing Generators (Experiment B: MixUp Augmentation - NUMPY FIX)...
MixUp Generators Ready.


In [4]:
# ==========================================================
# BLOCK 4 & 5: 3D RESNET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
clear_session()
gc.collect()

# 2. SOTA Training Configuration
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D ResNet Building Blocks
def conv3d_bn(x, filters, kernel_size, strides=(1,1,1), padding='same', activation=True):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    if activation:
        x = Activation('relu')(x)
    return x

def resnet_block(x, filters, strides=(1,1,1)):
    shortcut = x
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), strides=strides)
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), activation=False)
    if strides != (1,1,1) or x.shape[-1] != shortcut.shape[-1]:
        shortcut = conv3d_bn(shortcut, filters, kernel_size=(1,1,1), strides=strides, activation=False)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    x = conv3d_bn(video_input, 64, kernel_size=(7,7,7), strides=(1,2,2))
    x = tf.keras.layers.MaxPooling3D(pool_size=(1,3,3), strides=(1,2,2), padding='same')(x)
    
    x = resnet_block(x, 64)
    x = resnet_block(x, 64)
    x = resnet_block(x, 128, strides=(2,2,2)) 
    x = resnet_block(x, 128)
    x = resnet_block(x, 256, strides=(2,2,2))
    x = resnet_block(x, 256)
    x = resnet_block(x, 512, strides=(2,2,2))
    x = resnet_block(x, 512)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs=video_input, outputs=output, name='ResNet3D_18')

# 5. Initialize & Compile
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

# 6. Start Training
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 7. SOTA Evaluation & Inference Benchmarking
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

# Inference Latency/Throughput tracking
inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Specificity (True Negative Rate)
spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 8. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

I0000 00:00:1777023765.838195  500090 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13480 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: ResNet3D_18...
Epoch 1/50


I0000 00:00:1777023770.532848  500198 service.cc:152] XLA service 0x7fae24001880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777023770.532867  500198 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-24 15:42:50.764236: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777023771.689372  500198 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-24 15:42:55.487738: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller b

  1/906 ━━━━━━━━━━━━━━━━━━━━ 3:30:32 14s/step - accuracy: 0.2500 - auc: 0.6667 - loss: 1.0103

2026-04-24 15:43:00.255489: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 484 bytes spill stores, 484 bytes spill loads

I0000 00:00:1777023780.299784  500198 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 90s 84ms/step - accuracy: 0.4983 - auc: 0.6815 - loss: 0.7819 - val_accuracy: 0.6416 - val_auc: 0.8420 - val_loss: 0.6426
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 81ms/step - accuracy: 0.5422 - auc: 0.7269 - loss: 0.6817 - val_accuracy: 0.7765 - val_auc: 0.8754 - val_loss: 0.5669
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 72s 80ms/step - accuracy: 0.5764 - auc: 0.7590 - loss: 0.6325 - val_accuracy: 0.7566 - val_auc: 0.8613 - val_loss: 0.6425
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 72s 80ms/step - accuracy: 0.5872 - auc: 0.7732 - loss: 0.6105 - val_accuracy: 0.5022 - val_auc: 0.4062 - val_loss: 5.1421
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 72s 80ms/step - accuracy: 0.6024 - auc: 0.7800 - loss: 0.5946 - val_accuracy: 0.7965 - val_auc: 0.8679 - val_loss: 0.5469
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 72s 80ms/step - accuracy: 0.5831 - auc: 0.7934 - loss: 0.5827 - val_accuracy: 0.7301 - val_auc: 0.8266 - val_loss: 1.1132
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 6 & 7: I3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for I3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define I3D Building Blocks
# ----------------------------------------------------------
def conv3d_bn(x, filters, kernel_size, padding='same', strides=(1,1,1), name=None):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, 
               use_bias=False, kernel_regularizer=l2(1e-5), name=name)(x)
    x = BatchNormalization(scale=False)(x)
    x = Activation('relu')(x)
    return x

def inception_module(x, filters):
    f1x1, f3x3_reduce, f3x3, f5x5_reduce, f5x5, f_pool = filters

    branch1 = conv3d_bn(x, f1x1, (1, 1, 1))
    
    branch2 = conv3d_bn(x, f3x3_reduce, (1, 1, 1))
    branch2 = conv3d_bn(branch2, f3x3, (3, 3, 3))

    branch3 = conv3d_bn(x, f5x5_reduce, (1, 1, 1))
    branch3 = conv3d_bn(branch3, f5x5, (3, 3, 3))

    branch4 = MaxPooling3D((3, 3, 3), strides=(1, 1, 1), padding='same')(x)
    branch4 = conv3d_bn(branch4, f_pool, (1, 1, 1))

    x = Concatenate()([branch1, branch2, branch3, branch4])
    return x

def create_i3d_model(input_shape):
    video_input = Input(shape=input_shape)

    x = conv3d_bn(video_input, 64, (7, 7, 7), strides=(2, 2, 2))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)
    
    x = conv3d_bn(x, 64, (1, 1, 1))
    x = conv3d_bn(x, 192, (3, 3, 3))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = inception_module(x, [64, 96, 128, 16, 32, 32])
    x = inception_module(x, [128, 128, 192, 32, 96, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [192, 96, 208, 16, 48, 64])
    x = inception_module(x, [160, 112, 224, 24, 64, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [128, 128, 256, 24, 64, 64])
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='I3D_Inception')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_i3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_i3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for I3D.

Training Model: I3D_Inception...
Epoch 1/50
905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4757 - auc: 0.6554 - loss: 0.7539

2026-04-24 16:34:53.862713: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1045', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 61s 47ms/step - accuracy: 0.4975 - auc: 0.6883 - loss: 0.7329 - val_accuracy: 0.7588 - val_auc: 0.8473 - val_loss: 0.6218
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5433 - auc: 0.7372 - loss: 0.6751 - val_accuracy: 0.7633 - val_auc: 0.8516 - val_loss: 0.6154
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.5494 - auc: 0.7515 - loss: 0.6513 - val_accuracy: 0.7677 - val_auc: 0.8613 - val_loss: 0.6547
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.5795 - auc: 0.7783 - loss: 0.6337 - val_accuracy: 0.8009 - val_auc: 0.8852 - val_loss: 0.5699
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5905 - auc: 0.7868 - loss: 0.6265 - val_accuracy: 0.7876 - val_auc: 0.8839 - val_loss: 0.5817
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.5908 - auc: 0.7917 - loss: 0.6090 - val_accuracy: 0.8053 - val_auc: 0.8941 - val_loss: 0.5605
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 8 & 9: 3D DENSENET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    AveragePooling3D, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for 3D DenseNet.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D DenseNet Building Blocks
# ----------------------------------------------------------
def conv_block(x, growth_rate, name):
    x1 = BatchNormalization(name=name+'_0_bn')(x)
    x1 = Activation('relu', name=name+'_0_relu')(x1)
    x1 = Conv3D(4 * growth_rate, (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_1_conv')(x1)
    
    x1 = BatchNormalization(name=name+'_1_bn')(x1)
    x1 = Activation('relu', name=name+'_1_relu')(x1)
    x1 = Conv3D(growth_rate, (3, 3, 3), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_2_conv')(x1)
    
    x = Concatenate(name=name+'_concat')([x, x1])
    return x

def dense_block(x, blocks, name):
    # Reduced growth rate from 32 to 16 to prevent GPU OOM
    for i in range(blocks):
        x = conv_block(x, growth_rate=16, name=name + '_block' + str(i + 1))
    return x

def transition_block(x, reduction, name):
    x = BatchNormalization(name=name+'_bn')(x)
    x = Activation('relu', name=name+'_relu')(x)
    x = Conv3D(int(tf.keras.backend.int_shape(x)[-1] * reduction), (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_conv')(x)
    x = AveragePooling3D((2, 2, 2), strides=(2, 2, 2), name=name+'_pool')(x)
    return x

def create_densenet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(64, (7, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name='conv1_conv')(video_input)
    x = BatchNormalization(name='conv1_bn')(x)
    x = Activation('relu', name='conv1_relu')(x)
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same', name='pool1')(x)
    
    # Scaled down depth blocks for 3D Video memory limits
    x = dense_block(x, blocks=4, name='conv2')
    x = transition_block(x, 0.5, name='pool2')
    
    x = dense_block(x, blocks=8, name='conv3')
    x = transition_block(x, 0.5, name='pool3')
    
    x = dense_block(x, blocks=12, name='conv4')
    x = transition_block(x, 0.5, name='pool4')
    
    x = dense_block(x, blocks=8, name='conv5')
    
    x = BatchNormalization(name='bn')(x)
    x = Activation('relu')(x)
    x = GlobalAveragePooling3D(name='global_avg_pool')(x)
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid', name='fc_output')(x)

    return Model(inputs=video_input, outputs=output, name='DenseNet3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_densenet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_densenet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for 3D DenseNet.

Training Model: DenseNet3D_Lite...
Epoch 1/50


2026-04-24 17:01:21.376942: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_85', 28 bytes spill stores, 28 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 110s 65ms/step - accuracy: 0.5248 - auc: 0.7159 - loss: 0.6633 - val_accuracy: 0.6527 - val_auc: 0.8115 - val_loss: 0.6691
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.5585 - auc: 0.7574 - loss: 0.6352 - val_accuracy: 0.7566 - val_auc: 0.8485 - val_loss: 0.5724
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.5593 - auc: 0.7569 - loss: 0.6189 - val_accuracy: 0.7456 - val_auc: 0.8745 - val_loss: 0.5700
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.5715 - auc: 0.7669 - loss: 0.6122 - val_accuracy: 0.8053 - val_auc: 0.8747 - val_loss: 0.5406
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 57s 63ms/step - accuracy: 0.5767 - auc: 0.7773 - loss: 0.5997 - val_accuracy: 0.7876 - val_auc: 0.8824 - val_loss: 0.5352
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.5996 - auc: 0.7909 - loss: 0.5791 - val_accuracy: 0.7810 - val_auc: 0.8816 - val_loss: 0.5596
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 10 & 11: CONVNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, LayerNormalization, Dense, GlobalAveragePooling3D, 
    Dropout, Activation, Permute, Reshape, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.05 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvNeXt-3D Building Blocks
# ----------------------------------------------------------
class ConvNeXtBlock(tf.keras.layers.Layer):
    def __init__(self, dim, drop_path=0., **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.dwconv = Conv3D(dim, kernel_size=7, padding='same', groups=dim)
        self.norm = LayerNormalization(epsilon=1e-6)
        
        self.pwconv1 = Dense(4 * dim) 
        self.act = Activation('gelu')
        
        self.pwconv2 = Dense(dim)
        self.drop_path = Dropout(drop_path) if drop_path > 0. else tf.identity

    def call(self, inputs):
        input_tensor = inputs
        x = self.dwconv(inputs)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.drop_path(x)
        return input_tensor + x

def create_convnext3d_model(input_shape, depths=[3, 3, 9, 3], dims=[64, 128, 256, 512]):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(dims[0], kernel_size=(2, 4, 4), strides=(2, 4, 4), padding='valid', name='stem_conv')(video_input)
    x = LayerNormalization(epsilon=1e-6, name='stem_ln')(x)
    
    for i in range(4):
        dim = dims[i]
        depth = depths[i]
        
        for j in range(depth):
            x = ConvNeXtBlock(dim, name=f'stage{i}_block{j}')(x)
            
        if i < 3:
            x = LayerNormalization(epsilon=1e-6, name=f'stage{i}_downsample_ln')(x)
            x = Conv3D(dims[i+1], kernel_size=2, strides=2, padding='valid', name=f'stage{i}_downsample_conv')(x)

    x = GlobalAveragePooling3D()(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ConvNeXt3D_Nano')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convnext3d_model(input_shape, depths=[2, 2, 6, 2], dims=[48, 96, 192, 384])

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvNeXt-3D.

Training Model: ConvNeXt3D_Nano...
Epoch 1/50


2026-04-24 17:40:45.414482: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 20 bytes spill stores, 20 bytes spill loads

2026-04-24 17:40:45.579211: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_20', 324 bytes spill stores, 324 bytes spill loads

2026-04-24 17:40:45.593414: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4', 124 bytes spill stores, 124 bytes spill loads

2026-04-24 17:40:45.593685: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 408 bytes spill stores, 408 bytes spill loads

2026-04-24 17:40:45.595138: I external/local_xla/

  1/906 ━━━━━━━━━━━━━━━━━━━━ 4:33:39 18s/step - accuracy: 0.7500 - auc: 0.0000e+00 - loss: 0.3073

2026-04-24 17:40:55.701207: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_14', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_9', 4 bytes spill stores, 4 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 62s 48ms/step - accuracy: 0.3786 - auc: 0.5161 - loss: 0.7919 - val_accuracy: 0.5000 - val_auc: 0.6611 - val_loss: 0.6926
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.3711 - auc: 0.5074 - loss: 0.7222 - val_accuracy: 0.5000 - val_auc: 0.5175 - val_loss: 0.6941
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.3742 - auc: 0.4995 - loss: 0.7102 - val_accuracy: 0.5044 - val_auc: 0.6680 - val_loss: 0.6919
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.4639 - auc: 0.6303 - loss: 0.6672 - val_accuracy: 0.7301 - val_auc: 0.7992 - val_loss: 0.5825
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.5596 - auc: 0.7366 - loss: 0.5985 - val_accuracy: 0.7456 - val_auc: 0.8063 - val_loss: 0.5656
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.5731 - auc: 0.7362 - loss: 0.5824 - val_accuracy: 0.7544 - val_auc: 0.8343 - val_loss: 0.5646
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================================
# BLOCK 12 & 13: RESNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResNeXt-3D Building Blocks
# ----------------------------------------------------------
def grouped_conv3d(x, filters, kernel_size, strides=(1,1,1), padding='same', groups=8):
    try:
        return Conv3D(filters, kernel_size, strides=strides, padding=padding, 
                      groups=groups, use_bias=False, kernel_regularizer=l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = Conv3D(channels_per_group, kernel_size, strides=strides, padding=padding, 
                       use_bias=False, kernel_regularizer=l2(1e-5))(splits[i])
            group_list.append(g)
        return Concatenate(axis=-1)(group_list)

def resnext_block(x, filters, strides=(1,1,1), groups=8):
    shortcut = x
    bottleneck_width = filters // 2 

    x = Conv3D(bottleneck_width, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = grouped_conv3d(x, bottleneck_width, (3,3,3), strides=strides, padding='same', groups=groups)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnext3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Scaled down stem filters from 64 to 32
    x = Conv3D(32, (7,7,7), strides=(1,2,2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x)

    # Scaled down depths and set groups=8 to fit 16GB VRAM
    x = resnext_block(x, 64, groups=8)
    x = resnext_block(x, 64, groups=8)

    x = resnext_block(x, 128, strides=(2,2,2), groups=8)
    x = resnext_block(x, 128, groups=8)

    x = resnext_block(x, 256, strides=(2,2,2), groups=8)
    x = resnext_block(x, 256, groups=8)

    x = resnext_block(x, 512, strides=(2,2,2), groups=8)
    x = resnext_block(x, 512, groups=8)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ResNeXt3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnext3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNeXt-3D.


I0000 00:00:1777040751.088636 2171057 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13622 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: ResNeXt3D_Lite...
Epoch 1/50


I0000 00:00:1777040756.412271 2171164 service.cc:152] XLA service 0x78984c04ca10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777040756.412295 2171164 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-24 20:25:56.660955: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777040757.687531 2171164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/906 ━━━━━━━━━━━━━━━━━━━━ 43s 49ms/step - accuracy: 0.3333 - auc: 0.5407 - loss: 0.7292       

2026-04-24 20:26:06.456906: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1777040766.512123 2171164 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 63s 53ms/step - accuracy: 0.4834 - auc: 0.6645 - loss: 0.7829 - val_accuracy: 0.7699 - val_auc: 0.8421 - val_loss: 0.5906
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5011 - auc: 0.7091 - loss: 0.6944 - val_accuracy: 0.6350 - val_auc: 0.7470 - val_loss: 0.9431
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5588 - auc: 0.7487 - loss: 0.6378 - val_accuracy: 0.7765 - val_auc: 0.8760 - val_loss: 0.5845
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5698 - auc: 0.7655 - loss: 0.6171 - val_accuracy: 0.7633 - val_auc: 0.8563 - val_loss: 0.5632
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5756 - auc: 0.7699 - loss: 0.6050 - val_accuracy: 0.5796 - val_auc: 0.4742 - val_loss: 2.1587
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.5720 - auc: 0.7828 - loss: 0.5896 - val_accuracy: 0.8119 - val_auc: 0.8923 - val_loss: 0.5302
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 14 & 15: EFFICIENTNET-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply,
    Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for EfficientNet-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define EfficientNet-3D Building Blocks
# ----------------------------------------------------------
def get_activation(activation='swish'):
    return Activation(tf.nn.swish)

def squeeze_excitation_block(x, input_channels, squeeze_ratio=0.25):
    reduced_channels = max(1, int(input_channels * squeeze_ratio))
    
    se = GlobalAveragePooling3D()(x)
    se = Reshape((1, 1, 1, input_channels))(se)
    
    se = Dense(reduced_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = get_activation('swish')(se)
    
    se = Dense(input_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = Activation('sigmoid')(se)
    
    x = Multiply()([x, se])
    return x

def mbconv_block(x, input_filters, output_filters, kernel_size, strides, expand_ratio, use_se=True, drop_rate=0.0):
    shortcut = x 
    
    expanded_filters = input_filters * expand_ratio
    if expand_ratio != 1:
        x = Conv3D(expanded_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = get_activation('swish')(x)
    
    # Factorized (2+1)D approach to avoid 3D Dense Kernel explosion
    # 1. Spatial Convolution
    x = Conv3D(expanded_filters, (1, kernel_size, kernel_size), strides=(1, strides[1], strides[2]), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # 2. Temporal Convolution
    x = Conv3D(expanded_filters, (kernel_size, 1, 1), strides=(strides[0], 1, 1), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    if use_se:
        x = squeeze_excitation_block(x, expanded_filters)
    
    x = Conv3D(output_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    
    if strides == (1, 1, 1) and input_filters == output_filters:
        if drop_rate > 0:
            x = Dropout(drop_rate)(x)
        x = Add()([shortcut, x]) 
    
    return x

def create_efficientnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Stem
    x = Conv3D(16, 3, strides=(2, 2, 2), padding='same', use_bias=False, kernel_initializer='he_normal')(video_input)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # Stage 1
    x = mbconv_block(x, 16, 8, kernel_size=3, strides=(1,1,1), expand_ratio=1)
    
    # Stage 2
    x = mbconv_block(x, 8, 16, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 3
    x = mbconv_block(x, 16, 24, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 4
    x = mbconv_block(x, 24, 48, kernel_size=3, strides=(1,2,2), expand_ratio=2)
    x = mbconv_block(x, 48, 64, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Stage 5
    x = mbconv_block(x, 64, 96, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    x = mbconv_block(x, 96, 128, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Head
    x = Conv3D(512, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.2)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='EfficientNet3D_Factorized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_efficientnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_efficientnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for EfficientNet-3D.

Training Model: EfficientNet3D_Factorized...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 64s 51ms/step - accuracy: 0.5306 - auc: 0.7056 - loss: 0.6315 - val_accuracy: 0.7544 - val_auc: 0.8029 - val_loss: 0.6310
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5535 - auc: 0.7372 - loss: 0.6009 - val_accuracy: 0.7920 - val_auc: 0.8286 - val_loss: 0.5914
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.5762 - auc: 0.7587 - loss: 0.5842 - val_accuracy: 0.7588 - val_auc: 0.8292 - val_loss: 0.5436
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.5557 - auc: 0.7383 - loss: 0.5902 - val_accuracy: 0.7633 - val_auc: 0.8381 - val_loss: 0.5501
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.5651 - auc: 0.7507 - loss: 0.5816 - val_accuracy: 0.7942 - val_auc: 0.8757 - val_loss: 0.5245
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5844 - auc: 0.7574 - loss: 0.5703 - 

In [6]:
# ==========================================================
# BLOCK 16 & 17: VIVIT MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ViViT.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01 
LABEL_SMOOTHING = 0.1 
PROJECTION_DIM = 64  
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ViViT Components
# ----------------------------------------------------------
class TubeletEmbedding(layers.Layer):
    def __init__(self, embed_dim, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.projection = layers.Conv3D(
            filters=embed_dim, kernel_size=patch_size,
            strides=patch_size, padding="VALID", name="tubelet_proj"
        )
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, videos):
        projected_patches = self.projection(videos)
        flattened_patches = self.flatten(projected_patches)
        return flattened_patches

class PositionalEncoder(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        _, num_tokens, _ = input_shape
        self.position_embedding = self.add_weight(
            name="pos_embedding", shape=(num_tokens, self.embed_dim),
            initializer="glorot_uniform", trainable=True
        )

    def call(self, encoded_tokens):
        return encoded_tokens + self.position_embedding

def transformer_encoder_block(inputs, embed_dim, num_heads, ff_dim, dropout=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs]) 

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y) 
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    y = layers.Add()([y, x]) 
    return y

def create_vivit_model(input_shape):
    video_input = layers.Input(shape=input_shape)

    patches = TubeletEmbedding(embed_dim=PROJECTION_DIM, patch_size=(2, 16, 16))(video_input)
    encoded_patches = PositionalEncoder(embed_dim=PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        encoded_patches = transformer_encoder_block(
            encoded_patches, embed_dim=PROJECTION_DIM, 
            num_heads=NUM_HEADS, ff_dim=PROJECTION_DIM * 2, dropout=0.1
        )

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.GlobalAveragePooling1D()(representation)
    
    x = layers.Dropout(0.5)(representation)
    x = layers.Dense(128, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return Model(inputs=video_input, outputs=output, name="ViViT_Transformer")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_vivit_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_vivit_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ViViT.

Training Model: ViViT_Transformer...
Epoch 1/50


2026-04-24 21:26:48.527651: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_96', 60 bytes spill stores, 60 bytes spill loads

2026-04-24 21:26:48.574846: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_96', 60 bytes spill stores, 60 bytes spill loads

2026-04-24 21:26:48.579891: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_114', 808 bytes spill stores, 808 bytes spill loads

2026-04-24 21:26:48.598158: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_96', 440 bytes spill stores, 344 bytes spill loads

2026-04-24 21:26:48.621959: I external/loca

905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3650 - auc: 0.4927 - loss: 0.7830

2026-04-24 21:27:36.833699: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1286', 8 bytes spill stores, 8 bytes spill loads

2026-04-24 21:27:36.915901: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_38', 792 bytes spill stores, 792 bytes spill loads

2026-04-24 21:27:37.038363: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 192 bytes spill stores, 192 bytes spill loads

2026-04-24 21:27:37.042454: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_38', 980 bytes spill stores, 980 bytes spill loads

2026-04-24 21:27:37.049792: I external/loc

906/906 ━━━━━━━━━━━━━━━━━━━━ 57s 50ms/step - accuracy: 0.3698 - auc: 0.4999 - loss: 0.7545 - val_accuracy: 0.5398 - val_auc: 0.5887 - val_loss: 0.6900
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 47ms/step - accuracy: 0.3756 - auc: 0.5188 - loss: 0.7186 - val_accuracy: 0.5664 - val_auc: 0.5954 - val_loss: 0.6875
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.4123 - auc: 0.5236 - loss: 0.7083 - val_accuracy: 0.6173 - val_auc: 0.6091 - val_loss: 0.6838
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 47ms/step - accuracy: 0.3976 - auc: 0.5364 - loss: 0.7013 - val_accuracy: 0.5553 - val_auc: 0.6140 - val_loss: 0.6812
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 47ms/step - accuracy: 0.4106 - auc: 0.5465 - loss: 0.6981 - val_accuracy: 0.5929 - val_auc: 0.6216 - val_loss: 0.6788
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.4299 - auc: 0.5774 - loss: 0.6868 - val_accuracy: 0.5774 - val_auc: 0.6623 - val_loss: 0.6671
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 18 & 19: SLOWFAST MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D, AveragePooling3D, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for SlowFast.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define SlowFast Building Blocks
# ----------------------------------------------------------
def slowfast_block(x_slow, x_fast, filters, strides=(1,1,1)):
    # --- SLOW PATH (Spatial focus, 1x3x3) ---
    ys = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)

    if strides != (1,1,1) or x_slow.shape[-1] != filters*4:
        shortcut_s = Conv3D(filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
        shortcut_s = BatchNormalization()(shortcut_s)
    else:
        shortcut_s = x_slow
        
    # --- FAST PATH (Factorized Spatiotemporal Focus) ---
    fast_filters = max(1, filters // 4)  # Kept ratio tighter for Micro scale
    
    yf = Conv3D(fast_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    # Factorized (3,3,3) into (1,3,3) -> (3,1,1) to prevent XLA OOM crash
    yf = Conv3D(fast_filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters, (3,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    
    if strides != (1,1,1) or x_fast.shape[-1] != fast_filters*4:
        shortcut_f = Conv3D(fast_filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
        shortcut_f = BatchNormalization()(shortcut_f)
    else:
        shortcut_f = x_fast

    # --- LATERAL CONNECTION ---
    ys = Add()([ys, shortcut_s])
    ys = Activation('relu')(ys)
    
    yf = Add()([yf, shortcut_f])
    yf = Activation('relu')(yf)
    
    return ys, yf

def create_slowfast_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # --- Input Splitting ---
    x_fast = video_input
    x_slow = Lambda(lambda x: x[:, ::4, :, :, :], name='slow_slice')(video_input)
    
    # --- Stem ---
    # Slow Stem
    x_slow = Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_slow)
    x_slow = BatchNormalization()(x_slow)
    x_slow = Activation('relu')(x_slow)
    x_slow = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_slow)
    
    # Fast Stem (Factorized)
    x_fast = Conv3D(4, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = Conv3D(4, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_fast)
    
    # --- Stages ---
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 16)
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 32, strides=(1,2,2))
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 64, strides=(1,2,2))
    
    # --- Fusion & Head ---
    pool_slow = GlobalAveragePooling3D()(x_slow)
    pool_fast = GlobalAveragePooling3D()(x_fast)
    
    x = Concatenate()([pool_slow, pool_fast])
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='SlowFast_Micro')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_slowfast_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_slowfast_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for SlowFast.

Training Model: SlowFast_Micro...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 50ms/step - accuracy: 0.4873 - auc: 0.6694 - loss: 0.6746 - val_accuracy: 0.7765 - val_auc: 0.8480 - val_loss: 0.5413
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5392 - auc: 0.7410 - loss: 0.6069 - val_accuracy: 0.7788 - val_auc: 0.8671 - val_loss: 0.5234
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5566 - auc: 0.7618 - loss: 0.5877 - val_accuracy: 0.7788 - val_auc: 0.8476 - val_loss: 0.5452
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 47ms/step - accuracy: 0.5770 - auc: 0.7730 - loss: 0.5810 - val_accuracy: 0.7920 - val_auc: 0.8701 - val_loss: 0.5285
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5836 - auc: 0.7895 - loss: 0.5625 - val_accuracy: 0.7765 - val_auc: 0.8527 - val_loss: 0.5730
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 48ms/step - accuracy: 0.5869 - auc: 0.7790 - loss: 0.5631 - val_accuracy: 0.80

In [8]:
# ==========================================================
# BLOCK 20 & 21: R(2+1)D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for R(2+1)D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define R(2+1)D Building Blocks
# ----------------------------------------------------------
def conv2plus1d(x, filters, strides=(1,1,1)):
    inter_filters = filters 

    spatial_strides = (1, strides[1], strides[2])
    x = Conv3D(inter_filters, (1, 3, 3), strides=spatial_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    temporal_strides = (strides[0], 1, 1)
    x = Conv3D(filters, (3, 1, 1), strides=temporal_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    return x

def r2plus1d_block(x, filters, strides=(1,1,1)):
    shortcut = x
    
    x = conv2plus1d(x, filters, strides=strides)
    x = conv2plus1d(x, filters)
    
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', 
                          use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_r2plus1d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(45, (1, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv3D(64, (3, 1, 1), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = r2plus1d_block(x, 64)
    x = r2plus1d_block(x, 64)
    
    x = r2plus1d_block(x, 128, strides=(2, 2, 2))
    x = r2plus1d_block(x, 128)
    
    x = r2plus1d_block(x, 256, strides=(2, 2, 2))
    x = r2plus1d_block(x, 256)
    
    x = r2plus1d_block(x, 512, strides=(2, 2, 2))
    x = r2plus1d_block(x, 512)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='R2Plus1D_18')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_r2plus1d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_r2plus1d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for R(2+1)D.

Training Model: R2Plus1D_18...
Epoch 1/50


2026-04-24 22:40:18.809427: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.


906/906 ━━━━━━━━━━━━━━━━━━━━ 87s 77ms/step - accuracy: 0.4881 - auc: 0.6622 - loss: 0.8229 - val_accuracy: 0.7699 - val_auc: 0.8399 - val_loss: 0.7244
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 68s 75ms/step - accuracy: 0.5265 - auc: 0.7189 - loss: 0.7208 - val_accuracy: 0.7832 - val_auc: 0.8659 - val_loss: 0.6142
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 68s 76ms/step - accuracy: 0.5555 - auc: 0.7687 - loss: 0.6646 - val_accuracy: 0.8009 - val_auc: 0.8698 - val_loss: 0.5899
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 68s 75ms/step - accuracy: 0.5687 - auc: 0.7638 - loss: 0.6565 - val_accuracy: 0.7876 - val_auc: 0.8437 - val_loss: 0.6287
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 68s 75ms/step - accuracy: 0.5795 - auc: 0.7829 - loss: 0.6358 - val_accuracy: 0.7699 - val_auc: 0.8855 - val_loss: 0.6019
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 68s 75ms/step - accuracy: 0.5825 - auc: 0.7859 - loss: 0.6298 - val_accuracy: 0.7235 - val_auc: 0.8243 - val_loss: 0.6577
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [9]:
# ==========================================================
# BLOCK 22 & 23: X3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply, Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for X3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-5  # X3D is lightweight, needs less decay
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define X3D Building Blocks
# ----------------------------------------------------------
def swish(x):
    return Activation(tf.nn.swish)(x)

def se_block(x, filters, ratio=0.25):
    inputs = x
    x = GlobalAveragePooling3D()(x)
    x = Reshape((1, 1, 1, filters))(x)
    
    reduced_filters = max(1, int(filters * ratio))
    x = Dense(reduced_filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = swish(x)
    x = Dense(filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = Activation('sigmoid')(x)
    
    return Multiply()([inputs, x])

def x3d_bottleneck(x, filters, strides=(1,1,1), expansion_ratio=2.25):
    shortcut = x
    input_filters = x.shape[-1]
    expanded_filters = int(input_filters * expansion_ratio)

    x = Conv3D(expanded_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = Conv3D(expanded_filters, (3,3,3), strides=strides, padding='same', 
               groups=expanded_filters, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = se_block(x, expanded_filters)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or input_filters != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    return x 

def create_x3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(24, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = Conv3D(24, (5,1,1), strides=(1,1,1), padding='same', groups=24, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 48, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 96, strides=(2,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 192, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 192, expansion_ratio=2.25)
    
    x = Conv3D(432, (1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(2048, activation='relu')(x) 
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='X3D_M')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_x3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_x3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for X3D.

Training Model: X3D_M...
Epoch 1/50


2026-04-24 23:19:09.899185: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17039', 288 bytes spill stores, 288 bytes spill loads

2026-04-24 23:19:10.032184: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17039', 720 bytes spill stores, 720 bytes spill loads

2026-04-24 23:19:10.142339: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 24 bytes spill stores, 24 bytes spill loads

2026-04-24 23:19:10.311501: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 108 bytes spill stores, 108 bytes spill loads

2026-04-24 23:19:10.343902: I extern

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.5256 - auc: 0.7090 - loss: 0.6402

2026-04-24 23:21:12.366499: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 124 bytes spill stores, 124 bytes spill loads

2026-04-24 23:21:12.541083: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 716 bytes spill stores, 716 bytes spill loads

2026-04-24 23:21:12.547547: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 288 bytes spill stores, 288 bytes spill loads

2026-04-24 23:21:12.715427: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 28 bytes spill stores, 28 bytes spill loads

2026-04-24 23:21:12.722719: I external/loc

906/906 ━━━━━━━━━━━━━━━━━━━━ 142s 124ms/step - accuracy: 0.5566 - auc: 0.7431 - loss: 0.6016 - val_accuracy: 0.7080 - val_auc: 0.8260 - val_loss: 0.7084
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 110s 121ms/step - accuracy: 0.5847 - auc: 0.7796 - loss: 0.5476 - val_accuracy: 0.7942 - val_auc: 0.8739 - val_loss: 0.5035
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 108s 119ms/step - accuracy: 0.6043 - auc: 0.7994 - loss: 0.5359 - val_accuracy: 0.7367 - val_auc: 0.8604 - val_loss: 0.6361
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 109s 121ms/step - accuracy: 0.6118 - auc: 0.8030 - loss: 0.5246 - val_accuracy: 0.7788 - val_auc: 0.8763 - val_loss: 0.5132
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 109s 120ms/step - accuracy: 0.6159 - auc: 0.7972 - loss: 0.5228 - val_accuracy: 0.8319 - val_auc: 0.8991 - val_loss: 0.4696
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 109s 120ms/step - accuracy: 0.6325 - auc: 0.8077 - loss: 0.5115 - val_accuracy: 0.8296 - val_auc: 0.8942 - val_loss: 0.4750
Epoch 7/50
906/906 ━━━━━━━━

In [10]:
# ==========================================================
# BLOCK 24 & 25: C3D (MODERNIZED) MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for C3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define C3D Building Blocks
# ----------------------------------------------------------
def c3d_block(x, filters, count):
    for _ in range(count):
        x = Conv3D(filters, (3, 3, 3), activation='relu', padding='same', 
                   use_bias=False, kernel_regularizer=l2(1e-5))(x)
        x = BatchNormalization()(x)
    return x

def create_c3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = c3d_block(video_input, 64, 1)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 128, 1)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 256, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='C3D_Modernized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_c3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_c3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for C3D.

Training Model: C3D_Modernized...
Epoch 1/50


2026-04-25 00:22:42.730276: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2695', 96 bytes spill stores, 96 bytes spill loads

2026-04-25 00:22:43.789595: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-25 00:22:43.884171: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.5001 - auc: 0.6767 - loss: 0.8009

2026-04-25 00:25:48.805435: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_327', 64 bytes spill stores, 64 bytes spill loads

2026-04-25 00:25:49.194106: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-25 00:25:49.288974: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


906/906 ━━━━━━━━━━━━━━━━━━━━ 198s 200ms/step - accuracy: 0.5108 - auc: 0.6843 - loss: 0.7840 - val_accuracy: 0.7721 - val_auc: 0.8332 - val_loss: 0.6820
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 176s 195ms/step - accuracy: 0.5306 - auc: 0.7055 - loss: 0.7570 - val_accuracy: 0.6991 - val_auc: 0.8514 - val_loss: 0.7306
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 176s 194ms/step - accuracy: 0.5519 - auc: 0.7305 - loss: 0.7257 - val_accuracy: 0.7743 - val_auc: 0.8499 - val_loss: 0.6683
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 176s 194ms/step - accuracy: 0.5458 - auc: 0.7319 - loss: 0.7213 - val_accuracy: 0.7389 - val_auc: 0.8498 - val_loss: 0.6845
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 176s 194ms/step - accuracy: 0.5751 - auc: 0.7719 - loss: 0.6908 - val_accuracy: 0.7788 - val_auc: 0.8654 - val_loss: 0.6401
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 176s 194ms/step - accuracy: 0.5753 - auc: 0.7585 - loss: 0.6981 - val_accuracy: 0.7522 - val_auc: 0.8588 - val_loss: 0.6581
Epoch 7/50
906/906 ━━━━━━━━

In [11]:
# ==========================================================
# BLOCK 26 & 27: CNN-TRANSFORMER HYBRID TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for CNN-Transformer Hybrid.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01  # Transformers need higher decay
LABEL_SMOOTHING = 0.1
EMBED_DIM = 128      # Dimension for Transformer
NUM_HEADS = 4
TRANSFORMER_LAYERS = 2

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Model Components
# ----------------------------------------------------------
def create_cnn_feature_extractor(input_shape):
    cnn_input = layers.Input(shape=input_shape)
    
    # Stem
    x = layers.Conv2D(32, (7, 7), strides=2, padding='same', use_bias=False)(cnn_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding='same')(x)
    
    # Residual Block 1
    shortcut = x
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != 64:
        shortcut = layers.Conv2D(64, (1, 1), padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Residual Block 2 (Downsample)
    shortcut = x
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    shortcut = layers.Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Spatial Aggregation
    x = layers.GlobalAveragePooling2D()(x)
    
    return models.Model(inputs=cnn_input, outputs=x, name="cnn_extractor")

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.output_dim = output_dim

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        return inputs + embedded_positions

def create_cnn_transformer_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- 1. Spatial Feature Extraction (CNN) ---
    cnn_extractor = create_cnn_feature_extractor(input_shape[1:]) 
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- 2. Temporal Modeling (Transformer) ---
    x = layers.Dense(EMBED_DIM)(encoded_frames)
    x = PositionalEmbedding(sequence_length=NUM_FRAMES, output_dim=EMBED_DIM)(x)
    
    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=EMBED_DIM, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, x]) 

        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(EMBED_DIM * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(EMBED_DIM)(x3)
        x = layers.Add()([x3, x2]) 

    # --- 3. Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs=video_input, outputs=output, name="CNN_Transformer_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_cnn_transformer_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_cnn_transformer_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024)

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for CNN-Transformer Hybrid.

Training Model: CNN_Transformer_Hybrid...
Epoch 1/50


2026-04-25 02:39:48.873466: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 40 bytes spill stores, 48 bytes spill loads

2026-04-25 02:39:49.019795: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 184 bytes spill stores, 184 bytes spill loads

2026-04-25 02:39:49.388739: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17', 192 bytes spill stores, 192 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 66s 50ms/step - accuracy: 0.4440 - auc: 0.5983 - loss: 0.7765 - val_accuracy: 0.7323 - val_auc: 0.8245 - val_loss: 0.5935
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.4959 - auc: 0.6844 - loss: 0.6626 - val_accuracy: 0.6903 - val_auc: 0.8380 - val_loss: 0.5839
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 48ms/step - accuracy: 0.5237 - auc: 0.7183 - loss: 0.6308 - val_accuracy: 0.7588 - val_auc: 0.8572 - val_loss: 0.5380
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 42s 46ms/step - accuracy: 0.5334 - auc: 0.7458 - loss: 0.6145 - val_accuracy: 0.6903 - val_auc: 0.8439 - val_loss: 0.6176
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5433 - auc: 0.7465 - loss: 0.6053 - val_accuracy: 0.7721 - val_auc: 0.8550 - val_loss: 0.5521
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 43s 47ms/step - accuracy: 0.5436 - auc: 0.7581 - loss: 0.5968 - val_accuracy: 0.7920 - val_auc: 0.8744 - val_loss: 0.5230
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [12]:
# ==========================================================
# BLOCK 28 & 29: CONVLSTM HYBRID MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvLSTM.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvLSTM Components
# ----------------------------------------------------------
def create_cnn_backbone(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(512, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    return models.Model(inputs=inputs, outputs=x, name="cnn_backbone")

def create_convlstm_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    cnn = create_cnn_backbone(input_shape[1:])
    x = layers.TimeDistributed(cnn)(video_input)
    
    x = layers.ConvLSTM2D(
        filters=64, 
        kernel_size=(3, 3), 
        padding='same', 
        return_sequences=False, 
        dropout=0.2,
        recurrent_dropout=0.0 
    )(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="ConvLSTM_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convlstm_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convlstm_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvLSTM.

Training Model: ConvLSTM_Hybrid...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 69s 61ms/step - accuracy: 0.4735 - auc: 0.6504 - loss: 0.6625 - val_accuracy: 0.7522 - val_auc: 0.8405 - val_loss: 0.5584
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5472 - auc: 0.7346 - loss: 0.6058 - val_accuracy: 0.7699 - val_auc: 0.8711 - val_loss: 0.5264
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5676 - auc: 0.7614 - loss: 0.5800 - val_accuracy: 0.7965 - val_auc: 0.8781 - val_loss: 0.5104
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5706 - auc: 0.7847 - loss: 0.5700 - val_accuracy: 0.8230 - val_auc: 0.8843 - val_loss: 0.5018
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5996 - auc: 0.7979 - loss: 0.5506 - val_accuracy: 0.7876 - val_auc: 0.8833 - val_loss: 0.5065
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5847 - auc: 0.7941 - loss: 0.5507 - val_accuracy: 0.8

In [13]:
# ==========================================================
# BLOCK 30 & 31: TRANSFER LEARNING (MOBILENETV2 + LSTM) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Transfer Learning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_transfer_mobilenet_lstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ImageNet Pre-trained) ---
    base_cnn = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone (Critical for Transfer Learning)
    base_cnn.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='mobilenet_feature_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling (LSTM) ---
    x = layers.LSTM(256, return_sequences=False, dropout=0.3)(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_MobileNet_LSTM")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_transfer_mobilenet_lstm(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_mobilenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Transfer Learning.

Training Model: TL_MobileNet_LSTM...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 71s 58ms/step - accuracy: 0.5773 - auc: 0.7808 - loss: 0.5817 - val_accuracy: 0.8075 - val_auc: 0.8957 - val_loss: 0.4949
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.6065 - auc: 0.8174 - loss: 0.5222 - val_accuracy: 0.8119 - val_auc: 0.9100 - val_loss: 0.4643
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.6184 - auc: 0.8356 - loss: 0.5016 - val_accuracy: 0.8097 - val_auc: 0.9167 - val_loss: 0.5021
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.6244 - auc: 0.8459 - loss: 0.4944 - val_accuracy: 0.8186 - val_auc: 0.9129 - val_loss: 0.4632
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.6344 - auc: 0.8440 - loss: 0.4801 - val_accuracy: 0.8164 - val_auc: 0.9116 - val_loss: 0.4726
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.6407 - auc: 0.8539 - loss: 0.4792 - val_ac

In [14]:
# ==========================================================
# BLOCK 32 & 33: TRANSFER LEARNING (RESNET50 + ATTENTION) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNet50 Transfer.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 
ATTENTION_HEADS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_resnet_attention_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ResNet50 - Heavyweight) ---
    base_cnn = ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone
    base_cnn.trainable = False
    
    # Pooling immediately to save memory (2048 features per frame)
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='resnet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- Temporal Modeling (Attention) ---
    x = layers.LayerNormalization(epsilon=1e-6)(encoded_frames)
    
    attention_output = layers.MultiHeadAttention(
        num_heads=ATTENTION_HEADS, 
        key_dim=2048 // ATTENTION_HEADS, 
        dropout=0.1
    )(x, x)
    
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_ResNet50_Attention")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet_attention_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_resnet_attn_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNet50 Transfer.

Training Model: TL_ResNet50_Attention...
Epoch 1/50


2026-04-25 04:09:22.556485: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 560 bytes spill stores, 560 bytes spill loads

2026-04-25 04:09:22.598986: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 232 bytes spill stores, 232 bytes spill loads

2026-04-25 04:09:22.602849: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 240 bytes spill stores, 240 bytes spill loads

2026-04-25 04:09:22.642509: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 316 bytes spill stores, 920 bytes spill loads

2026-04-25 04:09:22.704462: I external/l

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.3695 - auc: 0.5115 - loss: 1.0863

2026-04-25 04:10:30.597423: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 12 bytes spill stores, 12 bytes spill loads

2026-04-25 04:10:30.702128: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 104 bytes spill stores, 104 bytes spill loads

2026-04-25 04:10:30.916747: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 4024 bytes spill stores, 4004 bytes spill loads

2026-04-25 04:10:31.051401: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 3876 bytes spill stores, 3868 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 117s 80ms/step - accuracy: 0.3698 - auc: 0.5086 - loss: 0.8391 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7253
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 61ms/step - accuracy: 0.3794 - auc: 0.4874 - loss: 0.7245 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7189
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.3744 - auc: 0.4880 - loss: 0.7212 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7139
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 61ms/step - accuracy: 0.3656 - auc: 0.5073 - loss: 0.7139 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7093
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 62ms/step - accuracy: 0.3780 - auc: 0.4977 - loss: 0.7182 - val_accuracy: 0.5000 - val_auc: 0.6704 - val_loss: 0.7302
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 61ms/step - accuracy: 0.3700 - auc: 0.5008 - loss: 0.7076 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7036
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [15]:
# ==========================================================
# BLOCK 34 & 35: FINE-TUNED DENSENET121 + BiLSTM & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Fine-Tuning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
FINE_TUNE_LR = 1e-5 
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Fine-Tuning Model
# ----------------------------------------------------------
def create_finetuned_densenet_bilstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- Backbone: DenseNet121 (ImageNet) ---
    base_cnn = DenseNet121(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # --- FINE-TUNING STRATEGY ---
    base_cnn.trainable = False
    
    # Unfreeze the last 50 layers
    for layer in base_cnn.layers[-50:]: 
        layer.trainable = True
        
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='densenet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling: Bidirectional LSTM ---
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=False, dropout=0.3))(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="DenseNet_BiLSTM_FineTuned")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_finetuned_densenet_bilstm(input_shape)

optimizer = optimizers.AdamW(learning_rate=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_finetuned_densenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Fine-Tuning.

Training Model: DenseNet_BiLSTM_FineTuned...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 255s 204ms/step - accuracy: 0.4365 - auc: 0.6105 - loss: 0.7051 - val_accuracy: 0.7522 - val_auc: 0.8523 - val_loss: 0.5925
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 162s 179ms/step - accuracy: 0.5061 - auc: 0.7175 - loss: 0.6404 - val_accuracy: 0.7965 - val_auc: 0.8833 - val_loss: 0.5205
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 162s 179ms/step - accuracy: 0.5425 - auc: 0.7575 - loss: 0.6167 - val_accuracy: 0.8119 - val_auc: 0.8972 - val_loss: 0.5071
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 179ms/step - accuracy: 0.5613 - auc: 0.7730 - loss: 0.5899 - val_accuracy: 0.8252 - val_auc: 0.9102 - val_loss: 0.4758
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 162s 179ms/step - accuracy: 0.5891 - auc: 0.7932 - loss: 0.5734 - val_accuracy: 0.8009 - val_auc: 0.9126 - val_loss: 0.5209
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 162s 179ms/step - accuracy: 0.5806 - auc: 0.8000 - loss: 0

In [4]:
# ==========================================================
# BLOCK 36 & 37: NANO3D MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation for attention-driven channel weighting."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_block(x, filters, strides=(1,1,1)):
    """Factorized Spatiotemporal Block with SE & Residual Connection."""
    shortcut = x
    
    # Spatial feature extraction
    x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Temporal feature extraction
    x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Stem ---
    x = layers.Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Conv3D(16, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Hierarchical Blocks ---
    # Block 1 (Low-level features)
    b1 = nano_block(x, 32)
    b1 = nano_block(b1, 32)
    
    # Block 2 (Mid-level features)
    b2 = nano_block(b1, 64, strides=(1,2,2))
    b2 = nano_block(b2, 64)
    
    # Block 3 (High-level features)
    b3 = nano_block(b2, 128, strides=(2,2,2))
    b3 = nano_block(b3, 128)
    
    # --- Multi-Scale Feature Fusion (DenseNet replacement) ---
    pool1 = layers.GlobalAveragePooling3D()(b1)
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    
    merged_features = layers.Concatenate()([pool1, pool2, pool3])
    
    # --- Classification Head ---
    x = layers.Dropout(0.4)(merged_features)
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D.


I0000 00:00:1777214564.295537 3824320 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13600 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: Nano3D...
Epoch 1/50


I0000 00:00:1777214568.842561 3824439 service.cc:152] XLA service 0x73546023aa40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777214568.842584 3824439 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-26 20:42:49.031412: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777214569.885489 3824439 cuda_dnn.cc:529] Loaded cuDNN version 91002


  4/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.7188 - auc: 0.7734 - loss: 0.6664 

I0000 00:00:1777214577.045494 3824439 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 63s 56ms/step - accuracy: 0.5395 - auc: 0.7273 - loss: 0.6408 - val_accuracy: 0.6881 - val_auc: 0.8377 - val_loss: 0.6230
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 49s 54ms/step - accuracy: 0.5483 - auc: 0.7546 - loss: 0.6081 - val_accuracy: 0.8009 - val_auc: 0.8825 - val_loss: 0.5327
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 49s 55ms/step - accuracy: 0.5759 - auc: 0.7718 - loss: 0.5786 - val_accuracy: 0.7788 - val_auc: 0.8621 - val_loss: 0.5589
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5955 - auc: 0.7834 - loss: 0.5668 - val_accuracy: 0.8164 - val_auc: 0.8843 - val_loss: 0.5359
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 50s 55ms/step - accuracy: 0.5919 - auc: 0.7857 - loss: 0.5613 - val_accuracy: 0.7920 - val_auc: 0.8731 - val_loss: 0.5311
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 50s 55ms/step - accuracy: 0.6021 - auc: 0.8005 - loss: 0.5435 - val_accuracy: 0.7854 - val_auc: 0.8873 - val_loss: 0.5548
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 36 & 37: NANO3D_EDGE TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D_Edge.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D_Edge Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation (Extremely low parameters)."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_edge_block(x, filters, strides=(1,1,1)):
    """Ultra-lightweight factorized block using pseudo-depthwise logic."""
    shortcut = x
    input_filters = x.shape[-1]
    
    # Spatial Pointwise Expansion (1x1x1)
    x = layers.Conv3D(filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Spatial Filtering (1x3x3) - Using groups to simulate depthwise
    groups_spatial = min(filters, 8) # Fallback to grouped if full depthwise isn't supported efficiently
    
    # Try grouped conv; if it fails (older TF), fall back to standard factorized
    try:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Temporal Filtering (3x1x1)
    try:
        x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
         x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or input_filters != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_edge_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Micro Stem ---
    x = layers.Conv3D(8, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Flat Hierarchical Blocks ---
    # Block 1 (8 -> 16 channels)
    b1 = nano_edge_block(x, 16)
    
    # Block 2 (16 -> 24 channels)
    b2 = nano_edge_block(b1, 24, strides=(1,2,2))
    
    # Block 3 (24 -> 32 channels)
    b3 = nano_edge_block(b2, 32, strides=(2,2,2))
    
    # Block 4 (32 -> 48 channels)
    b4 = nano_edge_block(b3, 48, strides=(2,2,2))
    
    # --- Multi-Scale Feature Fusion ---
    # Global average pooling on multi-scale outputs
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    pool4 = layers.GlobalAveragePooling3D()(b4)
    
    # Concatenate features (24 + 32 + 48 = 104 parameters fed to final head)
    merged_features = layers.Concatenate()([pool2, pool3, pool4])
    
    # --- Direct Classification Head (No dense layer bottleneck) ---
    x = layers.Dropout(0.4)(merged_features)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D_Edge')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_edge_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_edge_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.4f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D_Edge.

Training Model: Nano3D_Edge...
Epoch 1/50
  4/906 ━━━━━━━━━━━━━━━━━━━━ 43s 48ms/step - accuracy: 0.6042 - auc: 0.6444 - loss: 0.6393 

2026-04-26 21:23:37.593224: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 16 bytes spill stores, 16 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion_1', 16 bytes spill stores, 16 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 60s 55ms/step - accuracy: 0.5262 - auc: 0.7108 - loss: 0.6159 - val_accuracy: 0.7832 - val_auc: 0.8735 - val_loss: 0.5341
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5466 - auc: 0.7428 - loss: 0.5926 - val_accuracy: 0.7699 - val_auc: 0.8737 - val_loss: 0.5116
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5635 - auc: 0.7687 - loss: 0.5807 - val_accuracy: 0.7810 - val_auc: 0.8751 - val_loss: 0.5375
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.5704 - auc: 0.7639 - loss: 0.5655 - val_accuracy: 0.8053 - val_auc: 0.8782 - val_loss: 0.5075
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.6029 - auc: 0.7874 - loss: 0.5484 - val_accuracy: 0.7588 - val_auc: 0.8704 - val_loss: 0.5520
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5858 - auc: 0.7926 - loss: 0.5455 - val_accuracy: 0.8031 - val_auc: 0.8798 - val_loss: 0.5119
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 38 & 39: RESFORMER3D_MAX TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResFormer3D_Max.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResFormer3D_Max Architecture
# ----------------------------------------------------------
def res_block_3d(x, filters, strides=(1, 1, 1)):
    """Standard robust 3D Residual Block."""
    shortcut = x

    x = layers.Conv3D(filters, (3, 3, 3), strides=strides, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv3D(filters, (3, 3, 3), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    if strides != (1, 1, 1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1, 1, 1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout=0.3):
    """Standard Transformer Encoder Block."""
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs])

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y)
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([y, x])

def create_resformer_max(input_shape):
    video_input = layers.Input(shape=input_shape)

    # --- Deep 3D CNN Backbone ---
    x = layers.Conv3D(64, (5, 5, 5), strides=(1, 2, 2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = res_block_3d(x, 64)
    x = res_block_3d(x, 128, strides=(2, 2, 2))
    x = res_block_3d(x, 256, strides=(2, 2, 2))
    x = res_block_3d(x, 512, strides=(2, 2, 2))

    # --- Prepare for Transformer ---
    # We pool spatially but retain the temporal dimension
    x = layers.AveragePooling3D(pool_size=(1, x.shape[2], x.shape[3]))(x)
    
    # Reshape to (Batch, Time, Features) for Attention
    # Note: If NUM_FRAMES=16 and we downsampled time by factor of 8, Time=2
    time_steps = x.shape[1] 
    features = x.shape[-1]
    x = layers.Reshape((time_steps, features))(x)

    # --- Multi-Head Attention Bottleneck ---
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)

    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='ResFormer3D_Max')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resformer_max(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resformer_max.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResFormer3D_Max.

Training Model: ResFormer3D_Max...
Epoch 1/50


2026-04-26 22:00:16.077653: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 12 bytes spill stores, 12 bytes spill loads

2026-04-26 22:00:16.431692: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_46', 436 bytes spill stores, 436 bytes spill loads

2026-04-26 22:00:16.869313: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4', 444 bytes spill stores, 444 bytes spill loads

2026-04-26 22:00:16.871481: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 112 bytes spill stores, 112 bytes spill loads

2026-04-26 22:00:16.968945: I external/loca

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4540 - auc: 0.6148 - loss: 1.1945

2026-04-26 22:01:12.289629: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1003', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 66s 54ms/step - accuracy: 0.4688 - auc: 0.6415 - loss: 1.1541 - val_accuracy: 0.7257 - val_auc: 0.7864 - val_loss: 0.8088
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.4934 - auc: 0.6624 - loss: 1.0290 - val_accuracy: 0.7389 - val_auc: 0.8495 - val_loss: 0.7763
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.5477 - auc: 0.7394 - loss: 0.8820 - val_accuracy: 0.7854 - val_auc: 0.8730 - val_loss: 0.7558
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.5510 - auc: 0.7376 - loss: 0.8322 - val_accuracy: 0.8142 - val_auc: 0.8804 - val_loss: 0.7143
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5762 - auc: 0.7694 - loss: 0.7636 - val_accuracy: 0.7279 - val_auc: 0.8568 - val_loss: 0.7504
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.5993 - auc: 0.7842 - loss: 0.7451 - val_accuracy: 0.7965 - val_auc: 0.8947 - val_loss: 0.7167
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━